In [2]:
import csv

input_csv = "Online_Appendix_training_set.csv"
output_csv = "heartbeats.csv"

In [ ]:
with open(input_csv, 'r') as infile, open(output_csv, 'w', newline='') as outfile:
    reader = csv.DictReader(infile)

    fieldnames = ['location', 'repetitions']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        challenge = row['Challenge record name']
        repetitions = row['# Beat (automated algorithm)']

        writer.writerow({
            'location': challenge + '.wav',
            'repetitions': repetitions
        })


In [ ]:
#count number of items with repetitions < 9

with open(output_csv, 'r') as f:
    reader = csv.DictReader(f)
    count = sum(1 for row in reader if int(row['repetitions']) < 9)

print(f"Number of items with repetitions < 9: {count}")

In [ ]:
#histogram data collection
import csv
from collections import Counter

def count_repetition_frequencies(csv_path):
    repetitions = []

    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            repetitions.append(int(row["repetitions"]))

    freq = Counter(repetitions)

    return freq


# csv_file = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats.csv"
csv_file = "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"
hist_data = count_repetition_frequencies(csv_file)

for rep, count in sorted(hist_data.items()):
    print(f"{rep},{count}")


In [ ]:
#histogram of durations for items with repetitions < 9
import librosa
import csv

# csv_file = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats.csv"
# wavs_loc = "/scratch/local/hdd/hani/heartbeats/wav/"

csv_file = "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"
wavs_loc = "/scratch/local/hdd/hani/bbc_clocks/audio/"

durations = []
less_than_10 = 0
with open(csv_file, "r", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if int(row["repetitions"]) < 9:
            audio_path = wavs_loc + row["location"]
            y, sr = librosa.load(audio_path, sr=None)
            duration = librosa.get_duration(y=y, sr=sr)
            durations.append(duration)
            if duration <= 10:
                less_than_10 += 1

import matplotlib.pyplot as plt
plt.hist(durations, bins=20)
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Distribution of Audio Durations")
plt.show()


print(f"Number of audio files with duration <= 10 seconds: {less_than_10}")
print(f"Total number of audio files: {len(durations)}")

In [ ]:
#manual relabelling of heartbeats

import matplotlib.pyplot as plt
import librosa
import csv
import os
from numpy import linspace
from IPython.display import clear_output, Audio, display

csv_file = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats.csv"
wavs_loc = "/scratch/local/hdd/hani/heartbeats/wav/"
output_csv = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv"

total_target = 261 
start_file = "b0247.wav"
found_start = False 

already_processed_count = 0
if os.path.isfile(output_csv):
    with open(output_csv, "r") as f_check:
        already_processed_count = sum(1 for line in f_check) - 1
        if already_processed_count < 0: already_processed_count = 0

session_handled = 0 
session_skipped_empty = 0

file_exists = os.path.isfile(output_csv)

with open(csv_file, "r", newline="") as f:
    reader = list(csv.DictReader(f))
    
    with open(output_csv, "a", newline="") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=['location', 'repetitions', 'clear', 'double'])
        
        if not file_exists:
            writer.writeheader()

        for row in reader:
            if not found_start:
                if row['location'] == start_file:
                    found_start = True
                else:
                    continue 

            repetitions = int(row["repetitions"])
            
            if repetitions < 9:
                audio_path = wavs_loc + row["location"]
                try:
                    y, sr = librosa.load(audio_path, sr=None)
                except Exception as e:
                    print(f"Error loading {row['location']}: {e}")
                    continue

                duration = librosa.get_duration(y=y, sr=sr)
                if sr != 16000:
                    y = librosa.resample(y, orig_sr=sr, target_sr=16000)
                    sr = 16000

                fig, ax = plt.subplots(figsize=(10, 3))
                ax.plot(linspace(0, duration, len(y)), y)
                ax.set_title(f"File: {row['location']} | Reps: {repetitions}")
                plt.show()
                plt.pause(0.1)

                display(Audio(data=y*10.0, rate=sr))
                
                overall_progress = already_processed_count + session_handled
                print(f"Overall Progress: {overall_progress}/{total_target} files handled.")

                user_input = input(f"Enter count: ").strip().lower()
                
                if user_input == 'q':
                    break
                
                if user_input == "":
                    session_skipped_empty += 1
                    session_handled += 1
                    clear_output(wait=True)
                    continue

                try:
                    is_double = 1 if user_input.endswith('d') else 0
                    clean_input = user_input.replace('d', '')
                    
                    val = int(clean_input)
                    is_clear = 1 if val >= 0 else 0
                    final_reps = abs(val)
                    
                    writer.writerow({
                        'location': row['location'], 
                        'repetitions': final_reps, 
                        'clear': is_clear,
                        'double': is_double
                    })
                    
                    out_f.flush() 
                    session_handled += 1
                except ValueError:
                    print("Invalid input.")
                
                clear_output(wait=True)

print(f"--- Session Finished ---")
print(f"Files handled this session: {session_handled}")
print(f"Files skipped (empty input): {session_skipped_empty}")
print(f"Total labeled rows in CSV: {already_processed_count + (session_handled - session_skipped_empty)}")

In [ ]:
#count sorted csv number < 9, number of these clear, number of these double

sorted_csv = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv"

clear_count = 0
double_count = 0
total_count = 0

with open(sorted_csv, 'r') as f:
    reader = csv.DictReader(f)

    for row in reader:
        if int(row['repetitions']) < 9:
            total_count += 1
            if row['clear'] == '1':
                clear_count += 1
            if row['double'] == '1':
                double_count += 1

print(f"Total with repetitions < 9: {total_count}")
print(f"Number of these that are clear: {clear_count}")
print(f"Number of these that are double: {double_count}")
